00: Import Packages

In [49]:
import requests
import pandas as pd
import time
import ast
from google import genai
from google.genai import types
from datetime import datetime
from zoneinfo import ZoneInfo
from dateutil.relativedelta import relativedelta

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [17]:
# Tokens and URLs
gemini_token = ''

news_token = ''
news_url = 'https://newsapi.org/v2/everything'

forex_factory_url = 'https://nfs.faireconomy.media/ff_calendar_thisweek.json'

# Practice Acct No:
oanda_acct = ''
oanda_token = ''
oanda_url = 'api-fxpractice.oanda.com'

01: Pull Oanda Price Data

In [18]:
# Function definition to convert json to pandas dataframe
def json_to_pandas(json):
    json_file = json.json()
    price_json = json_file['candles']
    times = []
    granularity = []
    close_price, high_price, low_price, open_price = [], [], [], []

    for candle in price_json:
        dt_time = pd.to_datetime(
                datetime.fromisoformat(
                    candle['time'][:-1] + '+00:00')
                .astimezone(ZoneInfo('America/New_York')))
        times.append(dt_time)
        granularity.append(json_file['granularity'])
        close_price.append(float(candle['mid']['c']))
        high_price.append(float(candle['mid']['h']))
        low_price.append(float(candle['mid']['l']))
        open_price.append(float(candle['mid']['o']))

    df = pd.DataFrame({'datetime': pd.to_datetime(times).tz_localize(None),
                       'interval': granularity,
                       'close': close_price,
                       'high': high_price,
                       'low': low_price,
                       'open': open_price})
    df.index = pd.to_datetime(times).tz_localize(None)
    return df

In [28]:
def oanda_historical(currency,
                     oanda_token=oanda_token,
                     oanda_url=oanda_url,
                     oanda_acct=oanda_acct
                     ):
    li_interval = ['M5', 'M15', 'M30', 'H1', 'H4', 'D']
    li_lookback = [15, 30, 30, 30, 60, 90]
    df = pd.DataFrame()

    for interval, lookback in zip(li_interval, li_lookback):
        # Get historical prices
        start_time = datetime.now() - relativedelta(days=lookback)
        start_time = time.mktime(start_time.timetuple())
        end_time = time.mktime(datetime.now().timetuple()) - 1

        headers = {'Authorization': 'Bearer ' + oanda_token}
        query = {
            'from': str(start_time),
            'to': str(end_time),
            'granularity': interval,
            # 'alignmentTimezone': 'UTC'
        }
        candles_path = f'/v3/accounts/{oanda_acct}/instruments/{currency}/candles'
        response = requests.get('https://' + oanda_url + candles_path,
                                headers=headers,
                                params=query)
        print(f'Response {interval}: {response.status_code}')

        df_temp = json_to_pandas(response)
        df = pd.concat([df, df_temp], ignore_index=True)

    return df

02: Forex Factory

In [5]:
def forex_factory():
    response = requests.get('https://nfs.faireconomy.media/ff_calendar_thisweek.json')
    df = pd.DataFrame(response.json())
    df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
    return df

03: Gemini

In [6]:
def llm_chart_analysis(system_prompt, user_prompt):
    client = genai.Client(api_key=gemini_token)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[user_prompt],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0.1
        )
    )
    return response.text

In [26]:
# Rate limited
df_forex_factory = forex_factory()

In [97]:
# Specify Currency
currency = 'GBP_USD'
currency = 'GBP_JPY'
currency = 'GBP_NZD'
currency = 'GBP_AUD'
currency_prompt = currency.replace('_', '')

df_oanda_historical = oanda_historical(currency)
# Add in current time and price

system_prompt = f'''
You are a day trading expert in Forex Trading for a large financial organization. Your job is to analyze the {currency_prompt} and recommend profitable trades based on previous price action, upcoming news, and sentiment of the market. This includes an providing:
- entry price
- stop loss
- profit target

You are never to recommend any trades worth less than 25 pips.
In addition to your reasoning, please provide a list exactly like "TRADING_ARRAY:[entry price, stop loss, profit target]"
If no trade is recommended, please provide a list exactly like "TRADING_ARRAY:[0, 0, 0]".

All times are in US/Eastern.
'''

user_prompt = f'''
Please use the dataframe <df_forex_factory> to assist in your trade recommendations. This dataframe contains upcoming news for the week.
<df_forex_factory>
{df_forex_factory}
</df_forex_factory>

Please use the dataframe <df_oanda_historical> to assist in your trade recommendations. This dataframe contains historical price action in the form of candles (open, low, high, close) with timeframes specified in the `interval` field.
<df_oanda_historical>
{df_oanda_historical}
</df_oanda_historical>

Please take into account the current bid price, ask price, and spread in the <df_oanda_cba> dataframe.
<df_oanda_cba>
{df_oanda_cba}
</df_oanda_cba>
'''
response_llm = llm_chart_analysis(system_prompt, user_prompt)
print(response_llm)

Response M5: 200
Response M15: 200
Response M30: 200
Response H1: 200
Response H4: 200
Response D: 200
**GBPAUD Daily Analysis and Trade Recommendation:**

**Current Market Sentiment:**
The GBPAUD pair has shown a strong bullish momentum over the past few days, as evidenced by the upward trend on the H4 and Daily charts. The price has recently pushed to new highs, indicating underlying strength in the pair. However, the current market is exhibiting a very wide spread of 19.3 pips, which is a significant factor for day trading profitability.

**Upcoming News Analysis:**
Looking at the Forex Factory calendar, the most impactful news for GBP has already passed (CPI y/y on August 20 at 02:00:00 ET). The next high-impact news for GBP is the Flash Manufacturing and Services PMI on August 21 at 04:30:00 ET. For AUD, the upcoming news events are primarily low impact, with Flash Manufacturing and Services PMI scheduled for August 20 at 19:00:00 ET.

**Trade Recommendation Reasoning:**
Given the

04: Oanda Make Trade

In [36]:
def make_trade(currency,
               response_llm,
               lot_size):
    response_llm.find('TRADING_ARRAY')
    trading_array = ast.literal_eval(response_llm[response_llm.find('TRADING_ARRAY'):].replace('TRADING_ARRAY:', ''))
    entry = trading_array[0]
    stop_loss = trading_array[1]
    take_profit = trading_array[2]

    if trading_array == [0, 0, 0]:
        print('No trades were recommended.')
        return {
            'currency': currency,
            'error': 'No Trade'
        }

    if take_profit > entry:
        buy_sell = 'Buy'
        trend_sign = 1
    elif take_profit < entry:
        buy_sell = 'Sell'
        trend_sign = -1
    else:
        print('ERROR')
        return {
            'currency': currency,
            'error': 'ERROR'
        }

    print(f'Placing {buy_sell} Trade for {currency}')
    # Place Order
    units = int(trend_sign*lot_size)
    headers = {'Authorization': 'Bearer ' + oanda_token}
    orders_path = f'/v3/accounts/{oanda_acct}/orders'
    order = {'order': {
        'units': f'{units}',
        'instrument': f'{currency}',
        'timeInForce': 'FOK',
        'type': 'MARKET',
        'positionFill': 'DEFAULT'
    }}
    print(order)
    response_order = requests.post('https://' + oanda_url + orders_path, headers=headers, json=order)
    print(f'Order Response: {response_order}')
    trade_id = response_order.json()['orderFillTransaction']['tradeOpened']['tradeID']
    print(f'Order Number: {trade_id}')

    # Set Stoploss and Take Profit
    stop_loss_order = {'order': {
        'timeInForce': 'GTC',
        'price': f'{stop_loss}',
        'type': 'STOP_LOSS',
        'tradeID': trade_id
    }}
    take_profit_order = {'order': {
        'timeInForce': 'GTC',
        'price': f'{take_profit}',
        'type': 'TAKE_PROFIT',
        'tradeID': trade_id
    }}
    response_stop = requests.post('https://' + oanda_url + orders_path, headers=headers, json=stop_loss_order)
    print(f'Stop Loss Response: {response_stop}')
    response_profit = requests.post('https://' + oanda_url + orders_path, headers=headers, json=take_profit_order)
    print(f'Take Profit Response: {response_profit}')


    return {
        'currency': currency,
        'buy_sell': buy_sell,
        'trade_id': trade_id,
        'order': order,
        'order_response': f'{response_order.status_code}',
        'stop_loss': stop_loss_order,
        'stop_loss_response': f'{response_stop.status_code}',
        'take_profit': take_profit_order,
        'take_profit_response': f'{response_profit.status_code}',
    }

In [37]:
order = make_trade(currency,
                   response_llm,
                   1000)  # 100k is standard lot; 10k is mini lot; 1k is micro lot

# switch to limit order I think
# Save everything to google drive folder

Placing Sell Trade for GBP_NZD
{'order': {'units': '-1000', 'instrument': 'GBP_NZD', 'timeInForce': 'FOK', 'type': 'MARKET', 'positionFill': 'DEFAULT'}}
Order Response: <Response [201]>
Order Number: 116
Stop Loss Response: <Response [201]>
Take Profit Response: <Response [201]>


In [38]:
display(order)

{'currency': 'GBP_NZD',
 'buy_sell': 'Sell',
 'trade_id': '116',
 'order': {'order': {'units': '-1000',
   'instrument': 'GBP_NZD',
   'timeInForce': 'FOK',
   'type': 'MARKET',
   'positionFill': 'DEFAULT'}},
 'order_response': '201',
 'stop_loss': {'order': {'timeInForce': 'GTC',
   'price': '2.314',
   'type': 'STOP_LOSS',
   'tradeID': '116'}},
 'stop_loss_response': '201',
 'take_profit': {'order': {'timeInForce': 'GTC',
   'price': '2.29958',
   'type': 'TAKE_PROFIT',
   'tradeID': '116'}},
 'take_profit_response': '201'}

In [46]:
df_order = pd.DataFrame.from_dict(order)
df_order['reasoning'] = response_llm
df_order

,currency,buy_sell,trade_id,order,order_response,stop_loss,stop_loss_response,take_profit,take_profit_response,reasoning
order,GBP_NZD,Sell,116,"{'units': '-1000', 'instrument': 'GBP_NZD', 'timeInForce': 'FOK', 'type': 'MARKET', 'positionFill': 'DEFAULT'}",201,"{'timeInForce': 'GTC', 'price': '2.314', 'type': 'STOP_LOSS', 'tradeID': '116'}",201,"{'timeInForce': 'GTC', 'price': '2.29958', 'type': 'TAKE_PROFIT', 'tradeID': '116'}",201,"**GBPNZD Day Trading Recommendation**\n\n**Market Sentiment and Recent Price Action:**\nThe GBPNZD pair experienced a significant bullish surge following the RBNZ (Reserve Bank of New Zealand) Official Cash Rate announcement on August 19th, which likely indicated a more dovish stance or a rate cut than anticipated, weakening the NZD and strengthening GBPNZD. The price spiked from around 2.28500 to a high of 2.32160.\n\nHowever, after this sharp rally, the price action on the H4 and M5 charts indicates a strong pullback and potential profit-taking. The latest H4 candles show rejection from the highs and a clear bearish momentum developing, with the price currently trading around 2.30935. This suggests that the initial euphoria from the RBNZ news is fading, and a retracement of the large upward move is underway.\n\n**Upcoming News Analysis:**\nThe major NZD news (Official Cash Rate, Monetary Policy Statement, Rate Statement, Press Conference) has already passed and driven the recent price action.\nAn upcoming medium-impact event is the **RBNZ Gov Hawkesby Speaks** at 2025-08-20 16:10:00 NZT. This speech could introduce further volatility and either reinforce the current pullback (if dovish on NZD) or cause a rebound (if unexpectedly hawkish on NZD). Given the current technical setup, the market may continue its retracement into this event.\n\nFor GBP, there are high-impact Flash PMI data releases on August 21st, which are too far out to directly influence this immediate day trade but should be noted for future sessions.\n\n**Trade Recommendation:**\nBased on the strong initial rally followed by clear signs of a retracement on higher timeframes, a short position is recommended to capitalize on the ongoing pullback.\n\n* **Entry Price:** 2.30935 (Current market price, anticipating further retracement)\n* **Stop Loss:** 2.31400 (Placed above the recent H4 high of the pullback and the 23.6% Fibonacci retracement level of the main rally, providing a buffer against minor upward corrections.)\n* **Profit Target:** 2.29958 (This target aligns with the 61.8% Fibonacci retracement level of the recent significant bullish spike from 2.28596 to 2.32160. This is a common and logical retracement level after strong directional moves.)\n\n**Reasoning:**\nThe market has experienced an overextended move to the upside due to the RBNZ news. While the fundamental catalyst was strong, price action often corrects after such sharp movements. The current bearish candles on the H4 chart confirm this retracement. The chosen entry, stop loss, and profit target aim to capture a significant portion of this expected pullback while maintaining a favorable risk-to-reward ratio (approximately 1:2.1). The trade offers a potential profit of 97.7 pips with a risk of 46.5 pips, both exceeding the minimum 25 pips requirement.\n\nTRADING_ARRAY:[2.30935, 2.31400, 2.29958]"


In [94]:
def oanda_cba(currency,
              oanda_token=oanda_token,
              oanda_url=oanda_url,
              oanda_acct=oanda_acct
              ):

    headers = {'Authorization': 'Bearer ' + oanda_token}
    pricing_path = f'/v3/accounts/{oanda_acct}/pricing?instruments={currency}'
    response = requests.get('https://' + oanda_url + pricing_path,
                            headers=headers
                            )
    print(f'Current Pricing Response: {response.status_code}')

    cba_time = pd.to_datetime(
        datetime.fromisoformat(
            response.json()['time'][:-1] + '+00:00')
        .astimezone(ZoneInfo('America/New_York'))).tz_localize(None)
    bid = float(response.json()["prices"][0]['bids'][0]['price'])
    ask = float(response.json()["prices"][0]['asks'][0]['price'])
    spread = round(abs(bid - ask),5)

    df =  pd.DataFrame([{
        'currency': currency,
        'time': cba_time,
        'bid': bid,
        'ask': ask,
        'spread': spread
    }])

    return df

In [96]:
df_oanda_cba = oanda_cba(currency)

Current Pricing Response: 200
